# conv-output-shape — worked example 2: Conv output shape with dilation (effective kernel size)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-output-shape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Dilation inflates the kernel: a `K`-tap kernel with dilation `D` spans an *effective* size `K_eff = D*(K-1) + 1`. The full output-shape formula becomes `L_out = (L + 2P - D*(K-1) - 1) // S + 1`. Setting `D=1` recovers the familiar `(L + 2P - K)//S + 1`.

## Worked solution

We are given a 1-D signal of length `L=50`, kernel `K=3`, stride `S=1`, padding `P=2`, dilation `D=2`.

**Step 1 — compute the effective kernel size.** A dilated kernel inserts `D-1` gaps between taps. With `K=3` taps and `D=2`, the kernel covers `D*(K-1) + 1 = 2*2 + 1 = 5` input positions. So a dilation-2 size-3 kernel behaves spatially like a size-5 kernel for shape purposes.

**Step 2 — substitute into the formula.** Replace `K` with `K_eff` in `(L + 2P - K_eff)//S + 1`: `(50 + 2*2 - 5) // 1 + 1 = (50 + 4 - 5) // 1 + 1 = 49 // 1 + 1 = 49 + 1 = 50`.

**Step 3 — interpret.** Output length is 50 — unchanged from the input. The padding of 2 exactly compensates for the effective-kernel-5 shrinkage at stride 1, giving a SAME convolution.

**Why it works.** The convolution counts valid placements of the *effective* receptive field, not the raw tap count. Forgetting to expand by dilation is the most common conv-shape error. We confirm against `nn.Conv1d(..., dilation=2)`.

In [ ]:
def conv1d_outlen(L, K, S, P, D=1):
    K_eff = D * (K - 1) + 1
    return (L + 2 * P - K_eff) // S + 1


L, K, S, P, D = 50, 3, 1, 2, 2
pred_len = conv1d_outlen(L, K, S, P, D)
conv = t.nn.Conv1d(4, 7, kernel_size=K, stride=S, padding=P, dilation=D)
actual_len = conv(t.zeros(1, 4, L)).shape[-1]
print("predicted length:", pred_len)
print("actual length:   ", actual_len)
print("match:", pred_len == actual_len)